# RuBERT Fine-tuning for Affective State Detection

Fine-tune DeepPavlov/rubert-base-cased for 5-class emotion classification:
- **neutral** — normal patterns
- **frustrated** — short msgs, typos, errors
- **confused** — questions, uncertainty
- **engaged** — interest, follow-ups
- **confident** — assertions, correct answers

**Environment**: Colab Pro+ (T4/A100)

**Input**: `data/training/affect_labels.jsonl` (from `generate_affect_labels.py`)

**Output**: Fine-tuned model saved to `data/models/rubert_affect/`

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
import json
import os
import numpy as np
from pathlib import Path
from collections import Counter

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Configuration

In [ ]:
# Paths
LABELS_PATH = "data/training/affect_labels.jsonl"
OUTPUT_DIR = "data/models/rubert_affect"
CHECKPOINT_DIR = "data/models/rubert_affect_checkpoints"

# Model
MODEL_NAME = "DeepPavlov/rubert-base-cased"
MAX_LENGTH = 128  # Most student messages are short

# Training
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
NUM_EPOCHS = 10
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

# Labels
LABEL_MAP = {
    "neutral": 0,
    "frustrated": 1,
    "confused": 2,
    "engaged": 3,
    "confident": 4,
}
ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
NUM_LABELS = len(LABEL_MAP)

print(f"Labels: {LABEL_MAP}")
print(f"Num labels: {NUM_LABELS}")

## 2. Load and Split Data

In [ ]:
# Load labeled data
samples = []
with open(LABELS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        sample = json.loads(line)
        if sample["label"] in LABEL_MAP:
            samples.append(sample)

print(f"Total samples: {len(samples)}")

# Distribution
label_counts = Counter(s["label"] for s in samples)
print("\nLabel distribution:")
for label, count in sorted(label_counts.items()):
    print(f"  {label}: {count} ({100*count/len(samples):.1f}%)")

In [ ]:
# Split: 80% train, 10% val, 10% test
texts = [s["text"] for s in samples]
labels = [LABEL_MAP[s["label"]] for s in samples]

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

print(f"Train: {len(train_texts)}")
print(f"Val: {len(val_texts)}")
print(f"Test: {len(test_texts)}")

## 3. Tokenize and Create Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class AffectDataset(Dataset):
    """Dataset for affect classification."""

    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


train_dataset = AffectDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_dataset = AffectDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)
test_dataset = AffectDataset(test_texts, test_labels, tokenizer, MAX_LENGTH)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")
print(f"Test dataset: {len(test_dataset)} samples")

## 4. Model and Training

In [ ]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL,
    label2id=LABEL_MAP,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

In [ ]:
def compute_metrics(eval_pred):
    """Compute metrics for evaluation."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, predictions, average="macro")
    acc = accuracy_score(labels, predictions)
    return {"macro_f1": macro_f1, "accuracy": acc}


# Compute class weights for imbalanced data
label_counts_train = Counter(train_labels)
total_train = len(train_labels)
class_weights = torch.tensor(
    [total_train / (NUM_LABELS * label_counts_train.get(i, 1)) for i in range(NUM_LABELS)],
    dtype=torch.float32,
)
print(f"Class weights: {class_weights.tolist()}")

In [ ]:
# Custom Trainer with weighted loss
class WeightedTrainer(Trainer):
    """Trainer with class-weighted cross-entropy loss."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=3,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Starting training...")
train_result = trainer.train()
print(f"\nTraining complete!")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")

## 5. Evaluate on Test Set

In [ ]:
# Test evaluation
test_results = trainer.predict(test_dataset)
test_preds = np.argmax(test_results.predictions, axis=-1)
test_true = test_results.label_ids

print("=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(classification_report(
    test_true, test_preds,
    target_names=list(LABEL_MAP.keys()),
    digits=3,
))

macro_f1 = f1_score(test_true, test_preds, average="macro")
print(f"\nMacro F1: {macro_f1:.4f}")
print(f"Target: > 0.65")
print(f"{'PASS' if macro_f1 > 0.65 else 'NEEDS MORE DATA/TUNING'}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_true, test_preds)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title("Confusion Matrix — RuBERT Affect Detection")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")

labels_list = list(LABEL_MAP.keys())
ax.set_xticks(range(NUM_LABELS))
ax.set_yticks(range(NUM_LABELS))
ax.set_xticklabels(labels_list, rotation=45)
ax.set_yticklabels(labels_list)

# Add text annotations
for i in range(NUM_LABELS):
    for j in range(NUM_LABELS):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")

fig.colorbar(im)
plt.tight_layout()
plt.savefig("data/models/rubert_affect_confusion.png", dpi=150)
plt.show()
print("Confusion matrix saved.")

## 6. Training History

In [ ]:
# Plot training history
log_history = trainer.state.log_history

train_losses = [(h["step"], h["loss"]) for h in log_history if "loss" in h and "eval_loss" not in h]
eval_losses = [(h["step"], h["eval_loss"]) for h in log_history if "eval_loss" in h]
eval_f1s = [(h["step"], h["eval_macro_f1"]) for h in log_history if "eval_macro_f1" in h]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
if train_losses:
    steps, losses = zip(*train_losses)
    ax1.plot(steps, losses, label="Train Loss", alpha=0.7)
if eval_losses:
    steps, losses = zip(*eval_losses)
    ax1.plot(steps, losses, label="Val Loss", marker="o")
ax1.set_xlabel("Step")
ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

# F1
if eval_f1s:
    steps, f1s = zip(*eval_f1s)
    ax2.plot(steps, f1s, marker="o", color="green")
    ax2.axhline(y=0.65, color="red", linestyle="--", label="Target F1=0.65")
ax2.set_xlabel("Step")
ax2.set_ylabel("Macro F1")
ax2.set_title("Validation Macro F1")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("data/models/rubert_affect_training.png", dpi=150)
plt.show()

## 7. Save Model

In [ ]:
# Save best model
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save metadata
metadata = {
    "model_name": MODEL_NAME,
    "num_labels": NUM_LABELS,
    "label_map": LABEL_MAP,
    "id_to_label": ID_TO_LABEL,
    "max_length": MAX_LENGTH,
    "test_macro_f1": float(macro_f1),
    "test_accuracy": float(accuracy_score(test_true, test_preds)),
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "test_samples": len(test_dataset),
}
with open(os.path.join(OUTPUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Model saved to {OUTPUT_DIR}")
print(f"Metadata: {json.dumps(metadata, indent=2)}")

## 8. Quick Inference Test

In [ ]:
# Test inference
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=0 if torch.cuda.is_available() else -1,
)

test_messages = [
    "Не понимаю!!! Это невозможно!!",
    "Хм, а как это работает? Непонятно...",
    "Интересно, а что если попробовать по-другому?",
    "Я знаю ответ — это 42",
    "Ладно, следующий вопрос",
]

print("Inference test:")
print("=" * 50)
for msg in test_messages:
    result = classifier(msg)[0]
    print(f"  '{msg[:40]}...' → {result['label']} ({result['score']:.3f})")